In [1]:
import os
from glob import glob

import click
import os
import numpy as np
from scipy.spatial import KDTree
from scipy.signal import convolve2d
from spectral.io import envi

import sys
sys.path.append('/store/carroll/repos/SpectralUtil/spectral_util')
import spec_io, mosaic

import matplotlib.pyplot as plt
import time
from osgeo import osr
import pyproj
import logging
from tqdm import tqdm
from copy import deepcopy
osr.UseExceptions()

In [2]:
# define filepaths

home = '/store/carroll/col/data/2018/mosaic/'

output_file = '/store/carroll/col/data/2018/mosaic/neon_2018_mosaic_glt.tif'
input_file_list = '/store/carroll/col/data/2018/mosaic/file_lists/top_priority_isofit_obs.txt'
ignore_file_list = None
x_resolution = 1
y_resolution = None
target_extent_ul_lr = None
output_epsg = 32613
criteria_band = 5
criteria_mode = 'min'
n_cores = 1
max_distance = None

In [4]:
# new goal is to make extents
input_files = [x.strip() for x in open(input_file_list, 'r').readlines()]

In [8]:
gproj = osr.SpatialReference()
gproj.ImportFromEPSG(int(output_epsg))
wkt = gproj.ExportToWkt()
proj = pyproj.Proj(f"epsg:{output_epsg}")

if target_extent_ul_lr:
    ul_lr = target_extent_ul_lr # in output epsg projection
else:
    # Always gets this in 4326
    ul_lr = mosaic.get_ul_lr_from_files(input_files, get_resolution=False)
    # convert to output epsg
    ul = proj(ul_lr[0], ul_lr[1])
    lr = proj(ul_lr[2], ul_lr[3])
    ul_lr = [ul[0], ul[1], lr[0], lr[1]]

ul_lr

[316127.5833512874, 4324418.625222021, 337288.4827872183, 4297530.547926542]

In [11]:
# split into 9?
n = 3

xmin, ymax, xmax, ymin = ul_lr
dx = (xmax - xmin) / n
dy = (ymax - ymin) / n
dx, dy

(7053.633145310295, 8962.692431826144)

In [13]:
lis_ul_lr = []

for row in range(n):
    ul_y = ymax - row * dy
    lr_y = ymax - (row + 1) * dy
    for col in range(n):
        ul_x = xmin + col * dx
        lr_x = xmin + (col + 1) * dx
        lis_ul_lr.append([ul_x, ul_y, lr_x, lr_y])

lis_ul_lr

[[316127.5833512874, 4324418.625222021, 323181.2164965977, 4315455.932790195],
 [323181.2164965977, 4324418.625222021, 330234.849641908, 4315455.932790195],
 [330234.849641908, 4324418.625222021, 337288.4827872183, 4315455.932790195],
 [316127.5833512874, 4315455.932790195, 323181.2164965977, 4306493.2403583685],
 [323181.2164965977, 4315455.932790195, 330234.849641908, 4306493.2403583685],
 [330234.849641908, 4315455.932790195, 337288.4827872183, 4306493.2403583685],
 [316127.5833512874, 4306493.2403583685, 323181.2164965977, 4297530.547926542],
 [323181.2164965977, 4306493.2403583685, 330234.849641908, 4297530.547926542],
 [330234.849641908, 4306493.2403583685, 337288.4827872183, 4297530.547926542]]

In [26]:
# generate mosiac glt for each tile
import ast

ul_lr_grids = '/store/carroll/col/data/2018/mosaic/file_lists/ul_lr_grids.txt'
with open(ul_lr_grids, 'r') as f:
    ul_lr_grids = f.readlines()
ul_lr_grids = [ast.literal_eval(x.strip()) for x in ul_lr_grids]
ul_lr_grids

[[316127.5833512874, 4324418.625222021, 323181.2164965977, 4315455.932790195],
 [323181.2164965977, 4324418.625222021, 330234.849641908, 4315455.932790195],
 [330234.849641908, 4324418.625222021, 337288.4827872183, 4315455.932790195],
 [316127.5833512874, 4315455.932790195, 323181.2164965977, 4306493.2403583685],
 [323181.2164965977, 4315455.932790195, 330234.849641908, 4306493.2403583685],
 [330234.849641908, 4315455.932790195, 337288.4827872183, 4306493.2403583685],
 [316127.5833512874, 4306493.2403583685, 323181.2164965977, 4297530.547926542],
 [323181.2164965977, 4306493.2403583685, 330234.849641908, 4297530.547926542],
 [330234.849641908, 4306493.2403583685, 337288.4827872183, 4297530.547926542]]

In [30]:
for idx, ul_lr in enumerate(ul_lr_grids):
    print(idx, ul_lr)

0 [316127.5833512874, 4324418.625222021, 323181.2164965977, 4315455.932790195]
1 [323181.2164965977, 4324418.625222021, 330234.849641908, 4315455.932790195]
2 [330234.849641908, 4324418.625222021, 337288.4827872183, 4315455.932790195]
3 [316127.5833512874, 4315455.932790195, 323181.2164965977, 4306493.2403583685]
4 [323181.2164965977, 4315455.932790195, 330234.849641908, 4306493.2403583685]
5 [330234.849641908, 4315455.932790195, 337288.4827872183, 4306493.2403583685]
6 [316127.5833512874, 4306493.2403583685, 323181.2164965977, 4297530.547926542]
7 [323181.2164965977, 4306493.2403583685, 330234.849641908, 4297530.547926542]
8 [330234.849641908, 4306493.2403583685, 337288.4827872183, 4297530.547926542]


In [8]:
if y_resolution is not None and y_resolution > 0:
    logging.warning("y_resolution is set to a positive value, which is not common.  Unless this is being done very intentionally, stop, and make y negative.")
elif y_resolution is None:
    y_resolution = -1 * x_resolution
print(x_resolution, y_resolution)

if input_file_list.endswith(".nc"):
    input_files = [input_file_list]
else:
    input_files = [x.strip() for x in open(input_file_list, 'r').readlines()]

ignore_files = []
if ignore_file_list is not None:
    ignore_files = [x.strip() for x in open(ignore_file_list, 'r').readlines()]

gproj = osr.SpatialReference()
gproj.ImportFromEPSG(int(output_epsg))
wkt = gproj.ExportToWkt()
proj = pyproj.Proj(f"epsg:{output_epsg}")

1 -1


In [9]:
if target_extent_ul_lr:
    ul_lr = target_extent_ul_lr # in output epsg projection
else:
    # Always gets this in 4326
    ul_lr = mosaic.get_ul_lr_from_files(input_files, get_resolution=False)
    # convert to output epsg
    ul = proj(ul_lr[0], ul_lr[1])
    lr = proj(ul_lr[2], ul_lr[3])
    ul_lr = [ul[0], ul[1], lr[0], lr[1]]

if str(output_epsg)[0] == "4" and x_resolution > 1:
    raise ValueError(f"x_resolution is {x_resolution} (indicating meters), and EPSG is {output_epsg}.  Smells like lat/lon and UTM mismatch.  Terminating.")

print("Bounding box (ul_lr): " + str(ul_lr)) 

Bounding box (ul_lr): [316127.5833512874, 4324418.625222021, 337288.4827872183, 4297530.547926542]


In [12]:
trans

[316127.0833512874, 1, 0, 4324419.125222021, 0, -1]

In [11]:
trans = [ul_lr[0] - x_resolution/2., x_resolution, 0, 
         ul_lr[1] - y_resolution/2., 0, y_resolution]
meta = spec_io.GenericGeoMetadata(['GLT X', 'GLT Y', 'File Index', 'OBS val'], 
                                  projection=wkt, 
                                  geotransform=trans, 
                                  pre_orthod=True, 
                                  nodata_value=0)

glt = np.zeros(( int(np.ceil((ul_lr[3] - ul_lr[1]) / y_resolution)), 
                 int(np.ceil((ul_lr[2] - ul_lr[0]) / x_resolution)),
                 3), dtype=np.int32)
criteria = np.zeros((glt.shape[0], glt.shape[1]), dtype=np.float32)
criteria[...] = np.nan

y_grid_steps = np.arange(ul_lr[1], ul_lr[3] - trans[5]*0.01,trans[5])
x_grid_steps = np.arange(ul_lr[0], ul_lr[2] - trans[1]*0.01,trans[1])
y_grid, x_grid = np.meshgrid(y_grid_steps, 
                             x_grid_steps,
                             indexing='ij')

In [ ]:
# what if here I stop and - 
glt_x = x_grid
glt_y = y_grid
file_index = x_grid.copy()

In [15]:
glt.shape

(26889, 21161, 3)

In [17]:
_file = 0
file = input_files[0]
print(_file, file)

0 /store/carroll/col/data/2018/raw/L1/2018061214/NIS01_20180612_154959_rdn_obs_ort


In [19]:
nodata_fallback = -9999
with rasterio.open(file) as src2:
    # # --- use raster 1 as the reference grid ---
    # ref_crs = src1.crs
    # ref_transform = src1.transform
    # height, width = src1.height, src1.width

    # # read criteria band from raster 1
    # band1 = src1.read(criteria_band)
    # nodata1 = src1.nodata if src1.nodata is not None else nodata_fallback

    ref_crs = output_epsg
    ref_transform = trans
    height = glt.shape[0]
    width = glt.shape[1]
    band1 = glt
    
    # reproject criteria band from raster 2 to raster 1's grid
    band2 = np.full((height, width), nodata_fallback, dtype=band1.dtype)
    reproject(
        source=rasterio.band(src2, criteria_band),
        destination=band2,
        src_transform=src2.transform,
        src_crs=src2.crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        dst_nodata=nodata_fallback,
        resampling=Resampling.nearest,   # or bilinear, etc.
    )
    nodata2 = src2.nodata if src2.nodata is not None else nodata_fallback

TypeError: GDAL-style transforms have been deprecated.  This exception will be raised for a period of time to highlight potentially confusing errors, but will eventually be removed.

In [13]:
glt.shape, criteria.shape

((26889, 21161, 3), (26889, 21161))

In [7]:
y_grid.shape, x_grid.shape

((26889, 21161), (26889, 21161))

In [16]:
_file = 0
file = input_files[0]
print(_file, file)

0 /store/carroll/col/data/2018/raw/L1/2018061214/NIS01_20180612_154959_rdn_obs_ort


In [13]:
local_meta, obs = spec_io.load_data(file.strip(), lazy=True, load_glt=False, load_loc=True)
print(obs.shape)

(3562, 1631, 11)


In [14]:
loc_file = file.replace('obs_ort','ort_igm_ort')+'.hdr' # for NEON-formatted envi files
loc = envi.open(loc_file).open_memmap()[...,0:2]
print(loc.shape)

(3562, 1631, 2)


In [17]:
y_subgrid = loc[...,1]
x_subgrid = loc[...,0]
y_grid_minor, x_grid_minor, y_start, x_start = mosaic.get_subgrid_from_bounds(y_grid, x_grid, (np.min(y_subgrid), np.max(y_subgrid)), (np.min(x_subgrid), np.max(x_subgrid)))
y_grid_minor.shape, x_grid_minor.shape, y_start, x_start

((13476, 21161), (13476, 21161), 13413, 0)

In [18]:
main_grid_points = np.column_stack((y_grid_minor.ravel(), x_grid_minor.ravel()))

# Flatten the subgrid coordinates
subgrid_points = np.column_stack((y_subgrid.ravel(), x_subgrid.ravel()))

main_grid_points.shape, subgrid_points.shape

((285165636, 2), (5809622, 2))

In [20]:
# this is really fast
st = time.time()
tree = KDTree(subgrid_points)
print(f"Time to build tree: {time.time() - st}")

Time to build tree: 1.108036994934082


In [ ]:
st = time.time()
distances, indices = tree.query(main_grid_points, workers=1) # this is the thing that takes forever?
print(f"Time to querry tree: {time.time() - st}")

In [4]:
# trying my way....

obs_file_list = '/store/carroll/col/data/2018/mosaic/file_lists/top_priority_isofit_obs.txt'
criteria_band = 6   # 1-based index for rasterio; change as needed
nodata_value  = -9999  # change to your nodata if different

In [5]:
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling
